# Neural network #
Feed-forward network (PyTorch) on the same features and test period as `xgboost.ipynb`, tuned with Optuna on a validation period at the end of the training data, so the test set is only used once at the end.

Requires `pip install torch optuna`.

In [1]:
import time
import numpy as np
import pandas as pd
from pathlib import Path
import torch
from torch import nn
import optuna
from forecast_checks import LeakageChecker, ForecastValidator

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)

ModuleNotFoundError: No module named 'optuna'

In [2]:
DATA_DIR = Path("../data")
df = pd.read_csv(DATA_DIR / "dataset.csv", parse_dates=["date"])

In [3]:
#Same temporal split as xgboost.ipynb
cutoff = pd.to_datetime("2023-02-01")
train = df[df["date"] < cutoff - pd.Timedelta(days=6)]
test  = df[df["date"] >= cutoff]
LeakageChecker.check_split(train["date"], test["date"])

#Optuna tunes on the last 4 weeks of the training period (winter, like the test period), with the same 6-day gap
val_start = train["date"].max() - pd.Timedelta(days=27)
fit = train[train["date"] < val_start - pd.Timedelta(days=6)]
val = train[train["date"] >= val_start]
LeakageChecker.check_split(fit["date"], val["date"])
print(f"fit: {fit['date'].min().date()} - {fit['date'].max().date()}, val: {val['date'].min().date()} - {val['date'].max().date()}")

fit: 2022-04-07 - 2022-12-22, val: 2022-12-29 - 2023-01-25


In [4]:
feats = ["location", "hour", "weekday", "temperature",
         "consumption_6days_ago", "consumption_7days_ago", "consumption_14days_ago",
         "max_consumption_last_available_day", "min_consumption_last_available_day"]
LeakageChecker.check_feature_names(feats)
LAG_FEATS = [f for f in feats if "consumption" in f]
LOCATIONS = sorted(df["location"].unique())

#The network cannot take missing lags, so use the rows where every lag exists (same rows as the comparison in xgboost.ipynb)
train, fit, val, test = (d.dropna(subset=feats) for d in (train, fit, val, test))


class Scaler:
    """Scaling fitted on training data only: consumption relative to each location's mean, temperature standardized."""

    def __init__(self, data):
        #Oslo is about 6 times larger than Tromsø; relative consumption lets one network learn all locations
        self.level = data.groupby("location")["consumption"].mean()
        self.temp_mean, self.temp_std = data["temperature"].mean(), data["temperature"].std()

    def levels(self, data):
        return data["location"].map(self.level).to_numpy()

    def X(self, data):
        lags = data[LAG_FEATS].to_numpy() / self.levels(data)[:, None]
        temp = ((data["temperature"] - self.temp_mean) / self.temp_std).to_numpy()[:, None]
        onehot = [pd.get_dummies(pd.Categorical(data[col], categories=cats)).to_numpy()
                  for col, cats in (("location", LOCATIONS), ("hour", range(24)), ("weekday", range(7)))]
        return torch.tensor(np.hstack([lags, temp, *onehot]), dtype=torch.float32)

    def y(self, data):
        return torch.tensor(data["consumption"].to_numpy() / self.levels(data), dtype=torch.float32)

    def to_mw(self, data, pred):
        return pred * self.levels(data)

In [5]:
MAX_EPOCHS = 40
PATIENCE = 5


class MLP(nn.Module):
    def __init__(self, n_inputs, hidden, n_layers, dropout):
        super().__init__()
        layers = []
        for i in range(n_layers):
            layers += [nn.Linear(n_inputs if i == 0 else hidden, hidden), nn.ReLU(), nn.Dropout(dropout)]
        self.net = nn.Sequential(*layers, nn.Linear(hidden, 1))

    def forward(self, x):
        return self.net(x).squeeze(-1)


def predict(model, X):
    model.eval()
    with torch.no_grad():
        return model(X).numpy()


def train_model(params, X, y, epochs, val=None, trial=None):
    """Train with L1 loss (= MAE). With val = (X_val, y_val in MW, scaler, val data), stop early and return the best epoch."""
    torch.manual_seed(SEED)
    model = MLP(X.shape[1], params["hidden"], params["n_layers"], params["dropout"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    loss_fn = nn.L1Loss()
    best_mae, best_epoch = np.inf, 0
    for epoch in range(1, epochs + 1):
        model.train()
        #Shuffling rows is fine: every row already carries its own lag features
        for batch in torch.randperm(len(X)).split(params["batch_size"]):
            optimizer.zero_grad()
            loss_fn(model(X[batch]), y[batch]).backward()
            optimizer.step()
        if val is None:
            continue
        X_val, y_val, scaler, val_data = val
        mae = ForecastValidator.score(y_val, scaler.to_mw(val_data, predict(model, X_val)))["mae"]
        if mae < best_mae:
            best_mae, best_epoch = mae, epoch
        elif epoch - best_epoch >= PATIENCE:
            break
        if trial is not None:
            trial.report(mae, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
    return model, best_mae, best_epoch

In [6]:
#Tune on fit -> val; the scaler only sees the fit period
fit_scaler = Scaler(fit)
X_fit, y_fit = fit_scaler.X(fit), fit_scaler.y(fit)
val_set = (fit_scaler.X(val), val["consumption"], fit_scaler, val)


def objective(trial):
    params = {"n_layers": trial.suggest_int("n_layers", 1, 3),
              "hidden": trial.suggest_categorical("hidden", [32, 64, 128, 256]),
              "dropout": trial.suggest_float("dropout", 0.0, 0.3),
              "lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True),
              "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
              "batch_size": trial.suggest_categorical("batch_size", [256, 512, 1024])}
    _, mae, best_epoch = train_model(params, X_fit, y_fit, MAX_EPOCHS, val=val_set, trial=trial)
    trial.set_user_attr("best_epoch", best_epoch)
    return mae


start = time.time()
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED),
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5))
#Time budget: 4 minutes of tuning, the final fit below takes well under a minute
study.optimize(objective, timeout=240)
pruned = sum(t.state == optuna.trial.TrialState.PRUNED for t in study.trials)
print(f"{len(study.trials)} trials ({pruned} pruned) in {time.time() - start:.0f} s")
print("Best validation MAE:", study.best_value, "at epoch", study.best_trial.user_attrs["best_epoch"])
study.best_params

27 trials (14 pruned) in 242 s
Best validation MAE: 0.3030509773974433 at epoch 10


{'n_layers': 1,
 'hidden': 128,
 'dropout': 0.05806185716406084,
 'lr': 0.0015711453575407185,
 'weight_decay': 1.7565230804294371e-06,
 'batch_size': 256}

In [ ]:
#Refit on the whole training period with the best parameters and number of epochs, then predict the test period once
scaler = Scaler(train)
model, _, _ = train_model(study.best_params, scaler.X(train), scaler.y(train), study.best_trial.user_attrs["best_epoch"])
pred = scaler.to_mw(test, predict(model, scaler.X(test)))
print(f"Total training time: {time.time() - start:.0f} s")
test = test.assign(pred=pred)

In [8]:
#Same hours as the comparison in xgboost.ipynb (test rows where every lag exists), so the tables can be compared directly
validator = (ForecastValidator(test["consumption"], groups=test["location"])
             .add("Same hour last week", test["consumption_7days_ago"])
             .add("Neural network", test["pred"]))
validator.summary()

,mae,mae_pct_of_total,mse
Same hour last week,0.462299,8.505486,0.519314
Neural network,0.311378,5.728801,0.227612


In [9]:
validator.per_group()

mae  mae_pct_of_total       mse
model               group                                            
Same hour last week bergen       0.329611         10.375524  0.166252
                    helsingfors  0.148767          2.320853  0.079818
                    oslo         1.157653          9.694408  2.130099
                    stavanger    0.528136          9.570525  0.428968
                    tromsø       0.189610          9.762971  0.056069
                    trondheim    0.420232         11.601016  0.254981
Neural network      bergen       0.190583          5.999192  0.060084
                    helsingfors  0.178043          2.777564  0.070601
                    oslo         0.791754          6.630302  0.926972
                    stavanger    0.372768          6.755049  0.218662
                    tromsø       0.118867          6.120422  0.020691
                    trondheim    0.216344          5.972425  0.068771